# Step 3: A smarter baseline — user bias + movie bias

Our global-mean model scored RMSE ~1.0591 (full data) and ~1.0577
(hidden-only). That model treats every user and every movie identically.
Real ratings don't work that way — some users rate everything
generously, some movies are just better-loved than others.

**The idea:** instead of predicting the same number for everyone,
predict:

```
prediction = global_mean + user_bias + movie_bias
```

Where:
- `user_bias` = this user's average rating minus the global mean
  (positive if they rate higher than average, negative if lower)
- `movie_bias` = this movie's average rating minus the global mean
  (positive if it's generally well-loved, negative if not)

We compute both biases from the VISIBLE 80% only, then apply the
hide-and-predict trick again — same pattern as last time — to check
this smarter model actually performs better, and that it generalizes.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

DATA_DIR = "data"
SUBSAMPLE_SIZE = 500_000

train = pd.read_csv(f"{DATA_DIR}/train.csv")
if len(train) > SUBSAMPLE_SIZE:
    train = train.sample(n=SUBSAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Working with: {train.shape}")

Working with: (500000, 4)


In [2]:
visible, hidden = train_test_split(train, test_size=0.2, random_state=42)

print(f"Visible: {len(visible):,}   Hidden: {len(hidden):,}")

Visible: 400,000   Hidden: 100,000


## Compute global mean, user bias, and movie bias — from VISIBLE only

In [3]:
global_mean = visible["rating"].mean()

# Average rating per user, minus the global mean
user_bias = visible.groupby("userId")["rating"].mean() - global_mean

# Average rating per movie, minus the global mean
movie_bias = visible.groupby("movieId")["rating"].mean() - global_mean

print(f"Global mean: {global_mean:.4f}")
print(f"Users with a bias score:  {len(user_bias):,}")
print(f"Movies with a bias score: {len(movie_bias):,}")

Global mean: 3.5334
Users with a bias score:  109,506
Movies with a bias score: 16,816


## Predict on the hidden set

One wrinkle: the hidden set might contain a user or movie that never
appeared in visible at all (a "cold start"). For those, we have no
bias to look up — `.map()` returns NaN, so we fill those with 0
(meaning: fall back to just the global mean for that user/movie).

We also clip predictions to the valid [0.5, 5.0] range, since adding
two biases together can occasionally push a prediction outside the
real rating scale.

In [4]:
hidden = hidden.copy()

hidden["user_bias"]  = hidden["userId"].map(user_bias).fillna(0)
hidden["movie_bias"] = hidden["movieId"].map(movie_bias).fillna(0)
hidden["predicted"]  = global_mean + hidden["user_bias"] + hidden["movie_bias"]
hidden["predicted"]  = hidden["predicted"].clip(0.5, 5.0)

# Sanity check: how many hidden rows had a user or movie never seen in visible?
unseen_users  = (~hidden["userId"].isin(visible["userId"])).sum()
unseen_movies = (~hidden["movieId"].isin(visible["movieId"])).sum()
print(f"Hidden rows with a never-before-seen user:  {unseen_users}")
print(f"Hidden rows with a never-before-seen movie: {unseen_movies}")

hidden[["userId", "movieId", "rating", "predicted"]].head()

Hidden rows with a never-before-seen user:  10169
Hidden rows with a never-before-seen movie: 1560


,userId,movieId,rating,predicted
104241,34533,114060,4.5,3.807692
199676,61410,595,3.0,3.132381
140199,80317,1391,1.0,1.987501
132814,122913,27773,4.5,4.118919
408697,5319,1240,5.0,3.838118


## Check RMSE — did the smarter model actually help?

In [5]:
rmse_bias = np.sqrt(mean_squared_error(hidden["rating"], hidden["predicted"]))
rmse_mean_only = np.sqrt(mean_squared_error(hidden["rating"], np.full(len(hidden), global_mean)))

print(f"RMSE — global mean only (last step) : {rmse_mean_only:.4f}")
print(f"RMSE — user + movie bias (this step) : {rmse_bias:.4f}")
print(f"Improvement: {rmse_mean_only - rmse_bias:.4f}")

RMSE — global mean only (last step) : 1.0577
RMSE — user + movie bias (this step) : 1.0110
Improvement: 0.0467


## What we should expect

A clear drop in RMSE from the global-mean baseline. On real MovieLens
data, this typically takes RMSE from ~1.0 down to somewhere around
0.85–0.95 — a solid, meaningful improvement from a still very simple
model.

This is the same hide-and-predict pattern as last time, just applied
to a smarter model. Notice the workflow doesn't change — only the
prediction formula does. We'll keep reusing this exact structure for
every future model.

## Next step

This bias model is still "memorization with extra steps" — it doesn't
use any matrix factorization or learn relationships between users.
The next step is introducing **SVD++** (via the `surprise` library),
which is true collaborative filtering: it learns latent taste vectors
for users and movies rather than just averaging. We'll validate it
with this same hide-and-predict trick and compare its RMSE against
this bias model's score.